In [38]:
import pandas as pd
from astroquery.gaia import Gaia
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [116]:
data_dir = '~/Desktop/Data/'

In [117]:
#read in true sources
cs = pd.read_csv('../data/true_sources_final.csv')

In [118]:
#calc g mag error based on flux error 
cs['phot_g_mean_mag_error'] = (1.0857 * (cs['phot_g_mean_flux_error']/cs['phot_g_mean_flux'])).clip(lower=0.1) 

In [119]:
#get 2mass photometry
cache_path = Path('../data/tmass_query_results.csv')
if cache_path.exists():
    tmass = pd.read_csv(cache_path)
else:
    ids = (
        cs["source_id"]
        .dropna()
        .astype("int64")
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    batch_size = 500
    tmass_results = []

    for batch_number, start in enumerate(range(0, len(ids), batch_size), start=1):

        batch = ids[start:start + batch_size]
        id_string = ",".join(batch)

        query = f"""
            SELECT
                xmatch.source_id AS source_id,
                tmass.designation AS tmass_designation,

                tmass.j_m,
                tmass.j_msigcom AS j_err,

                tmass.h_m,
                tmass.h_msigcom AS h_err,

                tmass.ks_m,
                tmass.ks_msigcom AS ks_err,

                tmass.ph_qual

            FROM gaiadr3.tmass_psc_xsc_best_neighbour AS xmatch

            JOIN gaiadr3.tmass_psc_xsc_join AS xjoin
                ON xmatch.clean_tmass_psc_xsc_oid =
                xjoin.clean_tmass_psc_xsc_oid

            JOIN gaiadr1.tmass_original_valid AS tmass
                ON xjoin.original_psc_source_id =
                tmass.designation

            WHERE xmatch.source_id IN ({id_string})
            """

        print(f"Running batch {batch_number}...")

        try:
            job = Gaia.launch_job_async(query)
            result = job.get_results().to_pandas()
            tmass_results.append(result)

        except Exception as error:
            print(f"Batch {batch_number} failed:")
            print(error)
            print(query)
            raise

    tmass = pd.concat(tmass_results, ignore_index=True)
    tmass.to_csv(cache_path,index=False)
cs  = cs.merge(tmass,on='source_id',how='left')

In [ ]:
#get evolutionary state classifcations
rgb_rc = pd.read_csv(data_dir+'rgb_rc_cleaned_classification.csv')

In [121]:
cs = cs.merge(rgb_rc[['ID','ev_state']],left_on='fname',right_on='ID',how='left')
cs['nike_y'] = cs['numax']**0.75/cs['dnu']
cs['ev_state'] = cs['ev_state'].map({1: 'clump', 0: 'rgb'})
cs.loc[(cs['ev_state'] == 'rgb') & (cs['nike_y'] < 3.2), 'ev_state'] = float('nan')
cs.loc[(cs['ev_state'] == 'clump') & (cs['numax'] < 15), 'ev_state'] = 'agb'

In [122]:
#clean columns
cs = cs[['source_id', 'numax', 'numax_sig', 'dnu', 'dnu_sig', 'Teff', 's_Teff', 'feh', 's_feh', 
       'ra', 'dec', 
       'phot_g_mean_mag', 'phot_g_mean_mag_error', 'j_m', 'j_err', 'h_m', 'h_err', 'ks_m', 'ks_err', 'ev_state','ph_qual','syd_success']]

In [123]:
#masks to select different subsets

#do we have the phase
phase_known = cs["ev_state"].notna()

#is 2mass quality AAA (if so use)
use_tmass = (
    cs["ph_qual"]
    .fillna("")
    .str.strip()
    .eq("AAA")
)

#is the dnu measurement good (syd success == 2)
good_dnu = (
    pd.to_numeric(cs["syd_success"], errors="coerce").eq(2)
    & cs["dnu"].notna()
    & cs["dnu_sig"].notna()
)

In [124]:
cs = cs.iloc[:,:-2]

In [125]:
cs

,source_id,numax,numax_sig,dnu,dnu_sig,Teff,s_Teff,feh,s_feh,ra,dec,phot_g_mean_mag,phot_g_mean_mag_error,j_m,j_err,h_m,h_err,ks_m,ks_err,ev_state
0,2051287036390801024,97.22393,4.67503,8.78878,0.03617,4929.0,274.0,-0.07,0.38,290.182924,37.734885,17.159115,0.1,NaN,NaN,NaN,NaN,NaN,NaN,rgb
1,2051293702180102144,81.37265,1.37111,7.69527,0.08893,4630.0,106.0,-0.30,0.20,290.230496,37.816395,13.628399,0.1,11.922,0.021,11.346,0.020,11.250,0.017,rgb
2,2051294801692061952,31.95067,1.64749,4.16058,0.05572,4917.0,159.0,-0.33,0.25,290.185646,37.861626,15.426067,0.1,13.708,0.028,13.142,0.032,13.065,0.033,NaN
3,2051366682266279424,33.95322,0.86678,4.12243,0.02954,4628.0,116.0,-0.40,0.20,289.927063,38.283967,13.718153,0.1,11.949,0.023,11.336,0.019,11.259,0.020,rgb
4,2051664581201475840,5.49149,0.44858,1.04614,0.03536,4430.0,101.0,-0.30,0.20,293.288133,37.659449,11.570517,0.1,9.622,0.022,8.905,0.027,8.740,0.017,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2586,2129846626798007424,29.27067,1.81125,4.06402,0.08438,4874.0,159.0,-0.37,0.25,292.752538,49.854509,14.609172,0.1,13.041,0.041,12.422,NaN,12.385,NaN,clump
2587,2130924869747670528,28.11302,2.59228,4.05105,0.26378,4691.0,130.0,-0.83,0.25,288.613762,47.663902,14.878787,0.1,13.380,0.023,12.852,0.027,12.807,0.024,NaN
2588,2131013139914911616,26.34458,2.33074,3.68341,0.12152,4645.0,143.0,-0.13,0.24,288.421038,48.239535,15.124297,0.1,13.448,0.038,12.844,0.041,12.779,0.034,NaN
2589,2131992353798606208,15.57669,0.82501,2.14227,0.01828,4500.0,100.0,-0.10,0.20,285.060160,49.144449,11.209722,0.1,9.447,0.022,8.816,0.016,8.677,0.019,rgb


In [ ]:
cases = {
    "a": use_tmass & phase_known,
    "b": use_tmass & ~phase_known,
    "c": ~use_tmass & phase_known,
    "d": ~use_tmass & ~phase_known,
}

In [112]:
#four tables
#no numax correction
one = cs.copy()
two = cs[good_dnu].copy()

#numax correction
correction_factor = factor = -0.0588 * cs["feh"] + 0.949
corrected_cs = cs.copy()
corrected_cs['numax'] = factor*cs['numax']
corrected_cs['numax_sig'] = np.sqrt((factor*cs['numax_sig'])**2 + (0.0588*cs['numax']*cs['s_feh'])**2)

three = corrected_cs.copy()
four = corrected_cs[good_dnu].copy()


In [134]:
for case, mask in cases.items():
    one[mask].to_csv(f'../data/raw_{case}.csv',index=False,sep=" ", header=False,na_rep="-999.999")
    two[mask].to_csv(f'../data/raw_dnu_{case}.csv',index=False,sep=" ", header=False,na_rep="-999.999")
    three[mask].to_csv(f'../data/corrected_{case}.csv',index=False,sep=" ", header=False,na_rep="-999.999")
    four[mask].to_csv(f'../data/corrected_dnu_{case}.csv',index=False,sep=" ", header=False,na_rep="-999.999")


/var/folders/yw/flx0fz4d7ks6lxrv_kjhbg4h0000gp/T/ipykernel_22524/3156451741.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  two[mask].to_csv(f'../data/raw_dnu_{case}.csv',index=False,sep=" ", header=False,na_rep="-999.999")
/var/folders/yw/flx0fz4d7ks6lxrv_kjhbg4h0000gp/T/ipykernel_22524/3156451741.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  four[mask].to_csv(f'../data/corrected_dnu_{case}.csv',index=False,sep=" ", header=False,na_rep="-999.999")
/var/folders/yw/flx0fz4d7ks6lxrv_kjhbg4h0000gp/T/ipykernel_22524/3156451741.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  two[mask].to_csv(f'../data/raw_dnu_{case}.csv',index=False,sep=" ", header=False,na_rep="-999.999")
/var/folders/yw/flx0fz4d7ks6lxrv_kjhbg4h0000gp/T/ipykernel_22524/3156451741.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  four[mask].to_csv(f'../data/corrected_dnu_{case}.csv'